# Q1 Defense Experiments — OpenML Auto Notebook

This notebook needs **no external CSV files**.

It automatically downloads tabular classification datasets from **OpenML-CC18** using Kaggle internet.

Run order:
1. `RUN_STAGE = "build_openml"`
2. `RUN_STAGE = "external"`, `PART_ID = 1`
3. `RUN_STAGE = "external"`, `PART_ID = 2`
4. `RUN_STAGE = "small"`
5. `RUN_STAGE = "analyze"`

Final output:
`/kaggle/working/q1_defense_openml_outputs.zip`


In [ ]:

# =========================
# ONLY EDIT THIS CELL
# =========================

RUN_STAGE = "build_openml"
# Choose one:
# "build_openml"  -> downloads OpenML datasets and creates manifest
# "external"      -> full external replication; run twice with PART_ID 1 and 2
# "small"         -> mechanism + tuned sensitivity + MNAR sensitivity
# "analyze"       -> merge and summarize

PART_ID = 1
N_PARTS = 2

N_OPENML_DATASETS = 20       # recommended: 15-20
MAX_ROWS_PER_DATASET = 20000
LIMIT_TUNED_DATASETS = 8

OUT_ROOT = "/kaggle/working/q1_defense_openml"

print("RUN_STAGE:", RUN_STAGE)
print("PART:", PART_ID, "/", N_PARTS)
print("Output:", OUT_ROOT)


In [ ]:

# =========================
# SETUP
# =========================

import os, sys, json, time, warnings, shutil, math, re
from pathlib import Path
from typing import Any, Dict, List, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# OpenML package is only needed for getting the CC18 suite list.
# sklearn.fetch_openml will download the actual datasets.
try:
    import openml
except Exception:
    !pip install -q openml
    import openml

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("xgboost not available")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
    print("lightgbm not available")

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
print("Ready.")


In [ ]:

# =========================
# DATASET UTILITIES
# =========================

PMLB_MAIN_NAMES = {
    "adult", "mushroom", "allhypo", "clean2", "dna", "hypothyroid", "kr_vs_kp",
    "kr-vs-kp", "magic", "nursery", "optdigits", "pendigits", "phoneme", "satimage",
    "segmentation", "spambase", "splice", "tic_tac_toe", "tic-tac-toe", "vehicle",
    "vowel", "waveform_40", "mfeat_fourier", "mfeat_karhunen",
    "mfeat_morphological", "mfeat_zernike"
}

FALLBACK_OPENML_IDS = [
    31, 37, 44, 54, 15, 23, 29, 151, 1461, 1464, 1067, 1068, 1049, 1050, 1053,
    1480, 1485, 182, 188, 307, 458, 469, 6332, 40981, 40982, 40983, 40984, 40994
]

def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def safe_name(name):
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_")[:80]

def clean_columns(df):
    df = df.copy()
    df.columns = [safe_name(c) or f"col_{i}" for i, c in enumerate(df.columns)]
    # make unique
    seen = {}
    cols = []
    for c in df.columns:
        if c not in seen:
            seen[c] = 0
            cols.append(c)
        else:
            seen[c] += 1
            cols.append(f"{c}_{seen[c]}")
    df.columns = cols
    return df

def encode_target(y):
    le = LabelEncoder()
    return pd.Series(le.fit_transform(pd.Series(y).astype(str))).reset_index(drop=True)

def stratified_cap(X, y, max_rows=20000, seed=123):
    if max_rows is None or len(X) <= max_rows:
        return X.reset_index(drop=True), y.reset_index(drop=True)
    try:
        Xs, _, ys, _ = train_test_split(X, y, train_size=max_rows, random_state=seed, stratify=y)
        return Xs.reset_index(drop=True), ys.reset_index(drop=True)
    except Exception:
        rng = np.random.default_rng(seed)
        idx = rng.choice(np.arange(len(X)), size=max_rows, replace=False)
        return X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)

def dataset_complexity_ok(X):
    # Avoid very wide/high-cardinality datasets that make dense OHE too slow.
    if X.shape[1] > 120:
        return False, "too_many_raw_features"
    cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    total_card = 0
    for c in cat_cols:
        total_card += int(pd.Series(X[c]).nunique(dropna=True))
    if total_card > 350:
        return False, "too_high_categorical_cardinality"
    return True, "complexity_ok"

def validate_dataset(X, y, min_rows=500, min_minority=30, max_classes=20):
    if len(X) < min_rows:
        return False, "too_few_rows"
    if y.nunique() < 2:
        return False, "too_few_classes"
    if y.nunique() > max_classes:
        return False, "too_many_classes"
    if y.value_counts().min() < min_minority:
        return False, "minority_class_too_small"
    if X.shape[1] < 1:
        return False, "no_features"
    ok, reason = dataset_complexity_ok(X)
    if not ok:
        return False, reason
    return True, "valid"

def get_cc18_candidate_ids():
    try:
        suite = openml.study.get_suite(99)  # OpenML-CC18
        ids = list(dict.fromkeys(list(suite.data)))
        print("Loaded OpenML-CC18 suite 99 IDs:", len(ids))
        return ids
    except Exception as e:
        print("Could not load CC18 suite. Using fallback IDs.", str(e)[:200])
        return FALLBACK_OPENML_IDS

def build_openml_manifest():
    out_dir = ensure_dir(Path(OUT_ROOT) / "manifest")
    cache_dir = ensure_dir(Path(OUT_ROOT) / "openml_cache")

    manifest_path = out_dir / "openml_manifest_valid.csv"
    all_path = out_dir / "openml_manifest_all_checked.csv"

    rows = []
    selected = []
    seen_names = set()

    candidate_ids = get_cc18_candidate_ids()
    # add fallback after suite ids in case many are filtered
    candidate_ids = list(dict.fromkeys(candidate_ids + FALLBACK_OPENML_IDS))

    for data_id in candidate_ids:
        if len(selected) >= N_OPENML_DATASETS:
            break

        row = {"openml_id": data_id}
        try:
            data = fetch_openml(data_id=data_id, as_frame=True, parser="auto")
            name = str(data.details.get("name", f"openml_{data_id}"))
            name_safe = safe_name(name)
            lower_name = name_safe.lower()

            if lower_name in seen_names:
                row.update({"dataset_name": name_safe, "valid": False, "reason": "duplicate_name"})
                rows.append(row)
                continue

            if lower_name in PMLB_MAIN_NAMES:
                row.update({"dataset_name": name_safe, "valid": False, "reason": "skipped_main_pmlb_overlap"})
                rows.append(row)
                continue

            X = pd.DataFrame(data.data)
            y_raw = pd.Series(data.target)
            if y_raw is None or len(y_raw) == 0:
                row.update({"dataset_name": name_safe, "valid": False, "reason": "no_target"})
                rows.append(row)
                continue

            X = clean_columns(X)
            keep = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
            X = X[keep].copy()
            y = encode_target(y_raw)

            X, y = stratified_cap(X, y, MAX_ROWS_PER_DATASET)

            valid, reason = validate_dataset(X, y)
            row.update({
                "dataset_name": name_safe,
                "valid": valid,
                "reason": reason,
                "n_rows": int(len(X)),
                "n_features": int(X.shape[1]),
                "n_classes": int(y.nunique()),
                "majority_ratio": float(y.value_counts(normalize=True).max()) if y.nunique() else np.nan,
                "minority_count": int(y.value_counts().min()) if y.nunique() else 0,
            })

            if valid:
                df_cache = X.copy()
                df_cache["target"] = y.values
                csv_path = cache_dir / f"{name_safe}_{data_id}.csv"
                df_cache.to_csv(csv_path, index=False)

                row.update({
                    "csv_path": str(csv_path),
                    "target_col": "target",
                    "include": True,
                    "max_rows": MAX_ROWS_PER_DATASET,
                })
                selected.append(row.copy())
                seen_names.add(lower_name)
                print(f"Selected {len(selected):02d}: {name_safe} id={data_id} rows={len(X)} classes={y.nunique()}")
            else:
                print(f"Skip {name_safe} id={data_id}: {reason}")

        except Exception as e:
            row.update({"dataset_name": f"openml_{data_id}", "valid": False, "reason": "download_or_parse_failed", "error": str(e)[:300]})

        rows.append(row)

    pd.DataFrame(rows).to_csv(all_path, index=False)
    manifest = pd.DataFrame(selected)
    manifest.to_csv(manifest_path, index=False)

    print("\nSaved all checked:", all_path)
    print("Saved valid manifest:", manifest_path)
    print("Selected datasets:", len(manifest))
    display(manifest[["dataset_name", "openml_id", "n_rows", "n_features", "n_classes", "minority_count"]])
    return manifest_path

def load_from_manifest_row(row):
    df = pd.read_csv(row["csv_path"])
    target_col = row.get("target_col", "target")
    X = df.drop(columns=[target_col])
    y = pd.Series(df[target_col]).astype(int)
    return X.reset_index(drop=True), y.reset_index(drop=True)


In [ ]:

# =========================
# CORRUPTION UTILITIES
# =========================

def apply_label_noise(y, rate, seed):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).copy()
    flip = np.zeros(len(y), dtype=bool)
    if rate <= 0:
        return pd.Series(y), flip
    classes = np.unique(y)
    n_flip = int(round(rate * len(y)))
    if n_flip <= 0:
        return pd.Series(y), flip
    idx = rng.choice(np.arange(len(y)), size=n_flip, replace=False)
    for i in idx:
        choices = classes[classes != y[i]]
        if len(choices):
            y[i] = rng.choice(choices)
            flip[i] = True
    return pd.Series(y), flip

def apply_imbalance(X, y_observed, imbalance_level, seed):
    if str(imbalance_level) == "natural":
        idx = np.arange(len(X))
        return X.reset_index(drop=True), pd.Series(y_observed).reset_index(drop=True), idx

    rng = np.random.default_rng(seed)
    X = X.reset_index(drop=True)
    y = pd.Series(y_observed).reset_index(drop=True)
    target_ratio = float(imbalance_level)

    counts = y.value_counts()
    maj_class = counts.idxmax()
    maj_idx = y[y == maj_class].index.to_numpy()
    other_idx = y[y != maj_class].index.to_numpy()
    other_classes = [c for c in counts.index if c != maj_class]

    n_majority = len(maj_idx)
    desired_total = int(round(n_majority / target_ratio))
    desired_other = max(desired_total - n_majority, len(other_classes))
    desired_other = min(desired_other, len(other_idx))

    selected = []
    remaining = desired_other

    for c in other_classes:
        c_idx = y[y == c].index.to_numpy()
        if len(c_idx) > 0 and remaining > 0:
            selected.append(rng.choice(c_idx))
            remaining -= 1

    if remaining > 0:
        used = set(selected)
        avail = np.array([i for i in other_idx if i not in used])
        if len(avail) > 0:
            selected.extend(rng.choice(avail, size=min(remaining, len(avail)), replace=False).tolist())

    keep_idx = np.concatenate([maj_idx, np.array(selected, dtype=int)])
    rng.shuffle(keep_idx)
    return X.iloc[keep_idx].reset_index(drop=True), y.iloc[keep_idx].reset_index(drop=True), keep_idx

def numeric_codes_for_missing(X):
    Z = pd.DataFrame(index=X.index)
    for c in X.columns:
        if pd.api.types.is_numeric_dtype(X[c]):
            Z[c] = X[c]
        else:
            Z[c] = pd.Series(X[c]).astype("category").cat.codes.replace(-1, np.nan)
    return Z

def apply_mcar_missing(X, rate, seed):
    if rate <= 0:
        return X.copy()
    rng = np.random.default_rng(seed)
    return X.mask(rng.random(X.shape) < rate)

def apply_mar_missing(X, y, rate, seed):
    if rate <= 0:
        return X.copy()
    rng = np.random.default_rng(seed)
    Xout = X.copy()
    Z = numeric_codes_for_missing(Xout)

    counts = pd.Series(y).value_counts()
    minority_classes = set(counts[counts == counts.min()].index)
    is_minority = pd.Series(y).isin(minority_classes).to_numpy().astype(float)

    variances = Z.var(numeric_only=True).fillna(0)
    if len(variances) and variances.max() > 0:
        col = variances.idxmax()
        feature_signal = (Z[col] > Z[col].median()).fillna(False).to_numpy().astype(float)
    else:
        feature_signal = np.zeros(len(Xout))

    row_p = rate * (0.45 + 0.55 * is_minority + 0.35 * feature_signal)
    row_p = np.clip(row_p, 0, min(0.85, rate * 2.2))
    return Xout.mask(rng.random(Xout.shape) < row_p[:, None])

def apply_mnar_missing(X, rate, seed):
    if rate <= 0:
        return X.copy()
    rng = np.random.default_rng(seed)
    Xout = X.copy()
    Z = numeric_codes_for_missing(Xout)
    mask = np.zeros(Xout.shape, dtype=bool)
    for j, col in enumerate(Xout.columns):
        vals = pd.to_numeric(Z[col], errors="coerce")
        if vals.notna().sum() < 10 or vals.max() == vals.min():
            p = np.repeat(rate, len(Xout))
        else:
            rank = vals.rank(pct=True).fillna(0.5).to_numpy()
            p = rate * (0.4 + 1.2 * rank)
            p = np.clip(p, 0, min(0.85, rate * 2.0))
        mask[:, j] = rng.random(len(Xout)) < p
    return Xout.mask(mask)

def corrupt_train_test(X_train, y_train, X_test, y_test, label_noise, missing_rate, imbalance_level, seed, missing_mechanism="mcar"):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    y_test = pd.Series(y_test).reset_index(drop=True)

    y_obs, flip_mask = apply_label_noise(y_train, label_noise, seed + 10)
    X_sub, y_sub, keep_idx = apply_imbalance(X_train, y_obs, imbalance_level, seed + 20)

    if missing_mechanism == "mnar":
        X_sub = apply_mnar_missing(X_sub, missing_rate, seed + 30)
        X_test = apply_mnar_missing(X_test, missing_rate, seed + 40)
    elif missing_mechanism == "mar":
        X_sub = apply_mar_missing(X_sub, y_sub, missing_rate, seed + 30)
        X_test = apply_mar_missing(X_test, y_test, missing_rate, seed + 40)
    else:
        X_sub = apply_mcar_missing(X_sub, missing_rate, seed + 30)
        X_test = apply_mcar_missing(X_test, missing_rate, seed + 40)

    diag = {
        "train_rows_after_corruption": int(len(X_sub)),
        "n_label_flips_before_sampling": int(flip_mask.sum()),
    }

    return X_sub, pd.Series(y_sub).reset_index(drop=True), X_test, y_test, keep_idx, flip_mask, diag


In [ ]:

# =========================
# MODEL UTILITIES
# =========================

MODELS_9 = [
    "LogisticRegression", "LinearSVM", "KNN", "DecisionTree",
    "RandomForest", "ExtraTrees", "HistGradientBoosting", "XGBoost", "LightGBM"
]

def detect_cols(X):
    numeric = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical = [c for c in X.columns if c not in numeric]
    return numeric, categorical

def make_preprocessor(X, scale=False):
    numeric, categorical = detect_cols(X)
    if scale:
        num_pipe = Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())])
    else:
        num_pipe = Pipeline([("impute", SimpleImputer(strategy="mean"))])

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)])

    transformers = []
    if numeric:
        transformers.append(("num", num_pipe, numeric))
    if categorical:
        transformers.append(("cat", cat_pipe, categorical))

    return ColumnTransformer(transformers, remainder="drop")

def get_model(model_name, X, seed, n_classes, tuned_params=None):
    tuned_params = tuned_params or {}
    scale = model_name in ["LogisticRegression", "LinearSVM", "KNN"]
    pre = make_preprocessor(X, scale=scale)

    if model_name == "LogisticRegression":
        clf = LogisticRegression(max_iter=2000, random_state=seed, n_jobs=-1, **tuned_params)
    elif model_name == "LinearSVM":
        clf = LinearSVC(max_iter=6000, random_state=seed, **tuned_params)
    elif model_name == "KNN":
        params = {"n_neighbors": 5}; params.update(tuned_params)
        clf = KNeighborsClassifier(**params)
    elif model_name == "DecisionTree":
        clf = DecisionTreeClassifier(random_state=seed, **tuned_params)
    elif model_name == "RandomForest":
        clf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1, **tuned_params)
    elif model_name == "ExtraTrees":
        clf = ExtraTreesClassifier(n_estimators=200, random_state=seed, n_jobs=-1, **tuned_params)
    elif model_name == "HistGradientBoosting":
        clf = HistGradientBoostingClassifier(max_iter=150, random_state=seed, **tuned_params)
    elif model_name == "XGBoost":
        if not HAS_XGB:
            raise ImportError("xgboost unavailable")
        if n_classes == 2:
            clf = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                n_estimators=220, learning_rate=0.05, max_depth=6,
                                subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                                random_state=seed, n_jobs=-1, **tuned_params)
        else:
            clf = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", num_class=n_classes,
                                n_estimators=220, learning_rate=0.05, max_depth=6,
                                subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                                random_state=seed, n_jobs=-1, **tuned_params)
    elif model_name == "LightGBM":
        if not HAS_LGBM:
            raise ImportError("lightgbm unavailable")
        clf = LGBMClassifier(objective="binary" if n_classes == 2 else "multiclass",
                             n_estimators=220, learning_rate=0.05, num_leaves=31,
                             subsample=0.9, colsample_bytree=0.9,
                             random_state=seed, n_jobs=-1, verbose=-1, **tuned_params)
    else:
        raise ValueError(model_name)

    return Pipeline([("pre", pre), ("clf", clf)])

def compute_metrics(model, X_test, y_test):
    pred = model.predict(X_test)
    labels = np.unique(y_test)
    out = {
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(y_test, pred),
    }
    minority_class = pd.Series(y_test).value_counts().idxmin()
    recalls = recall_score(y_test, pred, average=None, labels=labels, zero_division=0)
    out["minority_recall"] = float(recalls[list(labels).index(minority_class)]) if minority_class in labels else np.nan
    return {k: float(v) for k, v in out.items()}

def full_grid():
    rows = []
    sid = 0
    for ln in [0.0, 0.1, 0.2, 0.3]:
        for mr in [0.0, 0.1, 0.2, 0.3]:
            for ir in ["natural", 0.7, 0.8, 0.9]:
                rows.append({"setting_id": sid, "label_noise": ln, "missing_rate": mr, "imbalance_level": str(ir), "imbalance_value": ir})
                sid += 1
    return rows

def key_settings():
    specs = [
        ("clean", 0.0, 0.0, "natural"),
        ("noise_only", 0.2, 0.0, "natural"),
        ("missing_only", 0.0, 0.2, "natural"),
        ("imbalance_only", 0.0, 0.0, 0.8),
        ("noise_missing", 0.2, 0.2, "natural"),
        ("noise_imbalance", 0.2, 0.0, 0.8),
        ("missing_imbalance", 0.0, 0.2, 0.8),
        ("all_three", 0.2, 0.2, 0.8),
    ]
    return [{"setting_id": i, "setting_name": name, "label_noise": ln, "missing_rate": mr, "imbalance_level": str(ir), "imbalance_value": ir}
            for i, (name, ln, mr, ir) in enumerate(specs)]

def save_row(row, path):
    path = Path(path)
    pd.DataFrame([row]).to_csv(path, mode="a", header=not path.exists(), index=False)

def make_run_key(dataset, seed, sid, model, suffix=""):
    return f"{dataset}__seed{seed}__setting{sid}__{model}{suffix}"


In [ ]:

# =========================
# EXTERNAL FULL-GRID RUN
# =========================

def run_external(manifest_path):
    out_dir = ensure_dir(Path(OUT_ROOT) / "external")
    raw_path = out_dir / f"raw_results_external_part_{PART_ID:02d}.csv"
    failed_path = out_dir / f"failed_runs_external_part_{PART_ID:02d}.csv"

    manifest = pd.read_csv(manifest_path).sort_values("dataset_name").reset_index(drop=True)
    parts = np.array_split(manifest.index.to_numpy(), N_PARTS)
    selected = manifest.iloc[parts[PART_ID - 1]].reset_index(drop=True)

    completed = set()
    if raw_path.exists():
        completed = set(pd.read_csv(raw_path)["run_key"].astype(str))
        print("Resuming completed:", len(completed))

    settings = full_grid()
    seeds = [0, 1, 2, 3, 4]

    print("Datasets in this part:", selected["dataset_name"].tolist())
    print("Planned fits:", len(selected) * len(seeds) * len(settings) * len(MODELS_9))

    for _, row in selected.iterrows():
        dataset = row["dataset_name"]
        print("\nDATASET:", dataset)

        try:
            X, y = load_from_manifest_row(row)
            valid, reason = validate_dataset(X, y)
            if not valid:
                print("Skip:", reason)
                continue
        except Exception as e:
            save_row({"dataset_name": dataset, "stage": "load", "error": str(e)[:500]}, failed_path)
            continue

        n_classes = int(y.nunique())

        for seed in seeds:
            try:
                Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
            except Exception as e:
                save_row({"dataset_name": dataset, "seed": seed, "stage": "split", "error": str(e)[:500]}, failed_path)
                continue

            for st in settings:
                sid = st["setting_id"]
                try:
                    Xc, yc, Xtc, ytc, keep_idx, flip_mask, diag = corrupt_train_test(
                        Xtr, ytr, Xte, yte,
                        st["label_noise"], st["missing_rate"], st["imbalance_value"],
                        seed=100000 + seed * 1000 + sid,
                        missing_mechanism="mcar"
                    )
                    if pd.Series(yc).nunique() < 2:
                        raise ValueError("less than two classes after corruption")
                except Exception as e:
                    save_row({"dataset_name": dataset, "seed": seed, "setting_id": sid, "stage": "corruption", "error": str(e)[:500]}, failed_path)
                    continue

                for model_name in MODELS_9:
                    run_key = make_run_key(dataset, seed, sid, model_name)
                    if run_key in completed:
                        continue

                    t0 = time.time()
                    try:
                        model = get_model(model_name, Xc, seed, n_classes)
                        model.fit(Xc, yc)
                        metrics = compute_metrics(model, Xtc, ytc)
                        result = {
                            "run_key": run_key, "dataset_name": dataset, "seed": seed,
                            "setting_id": sid, "model_name": model_name,
                            "label_noise": st["label_noise"], "missing_rate": st["missing_rate"],
                            "imbalance_level": st["imbalance_level"], "missing_mechanism": "mcar",
                            "order": "noise_then_imbalance",
                            "fit_eval_seconds": round(time.time() - t0, 4),
                        }
                        result.update(diag); result.update(metrics)
                        save_row(result, raw_path)
                        completed.add(run_key)

                        if len(completed) % 200 == 0:
                            print("Completed", len(completed), "runs")

                    except Exception as e:
                        save_row({"run_key": run_key, "dataset_name": dataset, "seed": seed, "setting_id": sid,
                                  "model_name": model_name, "stage": "fit_eval", "error": str(e)[:500]}, failed_path)

    print("Saved raw:", raw_path)


In [ ]:

# =========================
# SMALL EXPERIMENTS: MECHANISM + TUNED + MNAR
# =========================

def run_mechanism(manifest_path):
    out_dir = ensure_dir(Path(OUT_ROOT) / "mechanism")
    out_path = out_dir / "mechanism_diagnostics_noise_then_imbalance.csv"
    summary_path = out_dir / "mechanism_diagnostics_summary.csv"

    manifest = pd.read_csv(manifest_path)
    rows = []

    for _, row in manifest.iterrows():
        dataset = row["dataset_name"]
        print("Mechanism:", dataset)
        try:
            X, y = load_from_manifest_row(row)
            valid, _ = validate_dataset(X, y)
            if not valid:
                continue
        except Exception:
            continue

        for seed in [0, 1, 2, 3, 4]:
            try:
                Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
            except Exception:
                continue

            ytr = pd.Series(ytr).reset_index(drop=True)
            Xtr = Xtr.reset_index(drop=True)
            clean_counts = ytr.value_counts()
            clean_minority_class = clean_counts.idxmin()
            original_minority_count = int(clean_counts.min())

            for ln in [0.1, 0.2, 0.3]:
                y_noisy, flip_mask = apply_label_noise(ytr, ln, seed + 1000)
                for ir in [0.7, 0.8, 0.9]:
                    Xsub, ysub, keep_idx = apply_imbalance(Xtr, y_noisy, ir, seed + 2000)
                    keep_idx = np.asarray(keep_idx)
                    original_retained = ytr.iloc[keep_idx].reset_index(drop=True)
                    retained_flip = flip_mask[keep_idx]

                    clean_minority_retained = int((original_retained == clean_minority_class).sum())
                    mislabeled_retained = int(retained_flip.sum())
                    minority_mislabeled = int(((original_retained == clean_minority_class) & retained_flip).sum())

                    rows.append({
                        "dataset_name": dataset, "seed": seed, "label_noise": ln, "imbalance_level": ir,
                        "original_train_rows": int(len(ytr)),
                        "original_minority_count": original_minority_count,
                        "post_imbalance_rows": int(len(ysub)),
                        "clean_minority_retained_count": clean_minority_retained,
                        "clean_minority_retention_rate": clean_minority_retained / max(original_minority_count, 1),
                        "mislabeled_retained_count": mislabeled_retained,
                        "mislabeled_retained_ratio": mislabeled_retained / max(len(ysub), 1),
                        "minority_mislabeled_retained_count": minority_mislabeled,
                        "effective_clean_minority_ratio": clean_minority_retained / max(len(ysub), 1),
                        "observed_majority_ratio_after": float(pd.Series(ysub).value_counts(normalize=True).max()),
                        "observed_minority_count_after": int(pd.Series(ysub).value_counts().min()),
                    })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    summary = df.groupby(["label_noise", "imbalance_level"])[[
        "clean_minority_retention_rate", "effective_clean_minority_ratio",
        "mislabeled_retained_ratio", "minority_mislabeled_retained_count"
    ]].mean().reset_index()
    summary.to_csv(summary_path, index=False)
    print("Saved:", out_path)
    display(summary)

def tuning_grid(model_name):
    if model_name == "LogisticRegression":
        return {"clf__C": [0.1, 1.0, 10.0]}
    if model_name == "RandomForest":
        return {"clf__max_depth": [None, 8, 16], "clf__min_samples_leaf": [1, 3]}
    if model_name == "XGBoost":
        return {"clf__max_depth": [3, 6], "clf__learning_rate": [0.03, 0.08], "clf__n_estimators": [150, 250]}
    if model_name == "LightGBM":
        return {"clf__num_leaves": [15, 31], "clf__learning_rate": [0.03, 0.08], "clf__n_estimators": [150, 250]}
    return {}

def run_tuned(manifest_path):
    out_dir = ensure_dir(Path(OUT_ROOT) / "tuned")
    raw_path = out_dir / "raw_results_tuned_sensitivity.csv"
    failed_path = out_dir / "failed_runs_tuned_sensitivity.csv"
    params_path = out_dir / "selected_tuned_params.csv"

    manifest = pd.read_csv(manifest_path).head(int(LIMIT_TUNED_DATASETS))
    models = ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]
    settings = key_settings()

    completed = set()
    if raw_path.exists():
        completed = set(pd.read_csv(raw_path)["run_key"].astype(str))

    for _, row in manifest.iterrows():
        dataset = row["dataset_name"]
        print("\nTuned:", dataset)
        try:
            X, y = load_from_manifest_row(row)
            valid, _ = validate_dataset(X, y)
            if not valid:
                continue
        except Exception as e:
            save_row({"dataset_name": dataset, "stage": "load", "error": str(e)[:500]}, failed_path)
            continue

        n_classes = int(y.nunique())

        for seed in [0, 1, 2, 3, 4]:
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
            tuned_params = {}

            for m in models:
                try:
                    base = get_model(m, Xtr, seed, n_classes)
                    grid = tuning_grid(m)
                    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
                    search = GridSearchCV(base, grid, scoring="f1_macro", cv=cv, n_jobs=-1, error_score="raise")
                    search.fit(Xtr, ytr)
                    best = {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
                    tuned_params[m] = best
                    save_row({"dataset_name": dataset, "seed": seed, "model_name": m, "best_params": json.dumps(best)}, params_path)
                except Exception as e:
                    tuned_params[m] = {}
                    save_row({"dataset_name": dataset, "seed": seed, "model_name": m, "stage": "tuning", "error": str(e)[:500]}, failed_path)

            for st in settings:
                sid = st["setting_id"]
                try:
                    Xc, yc, Xtc, ytc, _, _, diag = corrupt_train_test(
                        Xtr, ytr, Xte, yte,
                        st["label_noise"], st["missing_rate"], st["imbalance_value"],
                        seed=500000 + seed * 1000 + sid,
                        missing_mechanism="mcar"
                    )
                except Exception as e:
                    save_row({"dataset_name": dataset, "seed": seed, "setting_id": sid, "stage": "corruption", "error": str(e)[:500]}, failed_path)
                    continue

                for m in models:
                    for variant in ["default", "tuned"]:
                        run_key = f"{dataset}__seed{seed}__setting{sid}__{m}__{variant}"
                        if run_key in completed:
                            continue
                        try:
                            params = tuned_params.get(m, {}) if variant == "tuned" else {}
                            model = get_model(m, Xc, seed, n_classes, tuned_params=params)
                            t0 = time.time()
                            model.fit(Xc, yc)
                            metrics = compute_metrics(model, Xtc, ytc)
                            result = {
                                "run_key": run_key, "dataset_name": dataset, "seed": seed,
                                "setting_id": sid, "setting_name": st["setting_name"],
                                "model_name": m, "variant": variant,
                                "label_noise": st["label_noise"], "missing_rate": st["missing_rate"],
                                "imbalance_level": st["imbalance_level"],
                                "fit_eval_seconds": round(time.time() - t0, 4),
                            }
                            result.update(diag); result.update(metrics)
                            save_row(result, raw_path)
                            completed.add(run_key)
                        except Exception as e:
                            save_row({"run_key": run_key, "dataset_name": dataset, "stage": "fit_eval", "error": str(e)[:500]}, failed_path)

    print("Saved tuned:", raw_path)

def run_mnar(manifest_path):
    out_dir = ensure_dir(Path(OUT_ROOT) / "mnar")
    raw_path = out_dir / "raw_results_mnar_sensitivity.csv"
    failed_path = out_dir / "failed_runs_mnar_sensitivity.csv"

    manifest = pd.read_csv(manifest_path)
    settings = key_settings()
    completed = set()
    if raw_path.exists():
        completed = set(pd.read_csv(raw_path)["run_key"].astype(str))

    for _, row in manifest.iterrows():
        dataset = row["dataset_name"]
        print("\nMNAR:", dataset)
        try:
            X, y = load_from_manifest_row(row)
            valid, _ = validate_dataset(X, y)
            if not valid:
                continue
        except Exception as e:
            save_row({"dataset_name": dataset, "stage": "load", "error": str(e)[:500]}, failed_path)
            continue

        n_classes = int(y.nunique())

        for seed in [0, 1, 2, 3, 4]:
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)

            for st in settings:
                sid = st["setting_id"]
                try:
                    Xc, yc, Xtc, ytc, _, _, diag = corrupt_train_test(
                        Xtr, ytr, Xte, yte,
                        st["label_noise"], st["missing_rate"], st["imbalance_value"],
                        seed=800000 + seed * 1000 + sid,
                        missing_mechanism="mnar"
                    )
                except Exception as e:
                    save_row({"dataset_name": dataset, "seed": seed, "setting_id": sid, "stage": "corruption", "error": str(e)[:500]}, failed_path)
                    continue

                for m in MODELS_9:
                    run_key = f"{dataset}__seed{seed}__setting{sid}__{m}__mnar"
                    if run_key in completed:
                        continue
                    try:
                        model = get_model(m, Xc, seed, n_classes)
                        t0 = time.time()
                        model.fit(Xc, yc)
                        metrics = compute_metrics(model, Xtc, ytc)
                        result = {
                            "run_key": run_key, "dataset_name": dataset, "seed": seed,
                            "setting_id": sid, "setting_name": st["setting_name"],
                            "model_name": m,
                            "label_noise": st["label_noise"], "missing_rate": st["missing_rate"],
                            "imbalance_level": st["imbalance_level"], "missing_mechanism": "mnar",
                            "fit_eval_seconds": round(time.time() - t0, 4),
                        }
                        result.update(diag); result.update(metrics)
                        save_row(result, raw_path)
                        completed.add(run_key)
                    except Exception as e:
                        save_row({"run_key": run_key, "dataset_name": dataset, "stage": "fit_eval", "error": str(e)[:500]}, failed_path)

    print("Saved MNAR:", raw_path)

def run_small(manifest_path):
    run_mechanism(manifest_path)
    run_tuned(manifest_path)
    run_mnar(manifest_path)


In [ ]:

# =========================
# ANALYSIS
# =========================

def metric_value(df, dataset, model, seed, ln, mr, ir, metric):
    q = df[
        (df["dataset_name"] == dataset) &
        (df["model_name"] == model) &
        (df["seed"] == seed) &
        (df["label_noise"].astype(float) == float(ln)) &
        (df["missing_rate"].astype(float) == float(mr)) &
        (df["imbalance_level"].astype(str) == str(ir))
    ]
    if len(q) == 0:
        return np.nan
    return float(q.iloc[0][metric])

def compute_interactions(raw, metric="macro_f1"):
    df = raw.copy()
    df["imbalance_level"] = df["imbalance_level"].astype(str)
    out = []

    noise_levels = sorted([x for x in df["label_noise"].astype(float).unique() if x > 0])
    missing_levels = sorted([x for x in df["missing_rate"].astype(float).unique() if x > 0])
    imbalance_levels = sorted([x for x in df["imbalance_level"].unique() if x != "natural"])

    for dataset in sorted(df["dataset_name"].unique()):
        for model in sorted(df["model_name"].unique()):
            for seed in sorted(df["seed"].unique()):
                p0 = metric_value(df, dataset, model, seed, 0, 0, "natural", metric)
                if pd.isna(p0):
                    continue

                for ln in noise_levels:
                    for mr in missing_levels:
                        pa = metric_value(df, dataset, model, seed, ln, 0, "natural", metric)
                        pb = metric_value(df, dataset, model, seed, 0, mr, "natural", metric)
                        pab = metric_value(df, dataset, model, seed, ln, mr, "natural", metric)
                        if not any(pd.isna(v) for v in [pa, pb, pab]):
                            da, db, dab = p0-pa, p0-pb, p0-pab
                            exp = da + db
                            out.append({"combination": "noise_missing", "dataset_name": dataset, "model_name": model, "seed": seed, "metric": metric,
                                        "actual_drop": dab, "expected_drop": exp, "interaction": dab-exp,
                                        "bounded_interaction": dab-min(exp,p0), "relative_clean_interaction": (dab-exp)/(p0+0.02)})

                for ln in noise_levels:
                    for ir in imbalance_levels:
                        pa = metric_value(df, dataset, model, seed, ln, 0, "natural", metric)
                        pb = metric_value(df, dataset, model, seed, 0, 0, ir, metric)
                        pab = metric_value(df, dataset, model, seed, ln, 0, ir, metric)
                        if not any(pd.isna(v) for v in [pa, pb, pab]):
                            da, db, dab = p0-pa, p0-pb, p0-pab
                            exp = da + db
                            out.append({"combination": "noise_imbalance", "dataset_name": dataset, "model_name": model, "seed": seed, "metric": metric,
                                        "actual_drop": dab, "expected_drop": exp, "interaction": dab-exp,
                                        "bounded_interaction": dab-min(exp,p0), "relative_clean_interaction": (dab-exp)/(p0+0.02)})

                for mr in missing_levels:
                    for ir in imbalance_levels:
                        pa = metric_value(df, dataset, model, seed, 0, mr, "natural", metric)
                        pb = metric_value(df, dataset, model, seed, 0, 0, ir, metric)
                        pab = metric_value(df, dataset, model, seed, 0, mr, ir, metric)
                        if not any(pd.isna(v) for v in [pa, pb, pab]):
                            da, db, dab = p0-pa, p0-pb, p0-pab
                            exp = da + db
                            out.append({"combination": "missing_imbalance", "dataset_name": dataset, "model_name": model, "seed": seed, "metric": metric,
                                        "actual_drop": dab, "expected_drop": exp, "interaction": dab-exp,
                                        "bounded_interaction": dab-min(exp,p0), "relative_clean_interaction": (dab-exp)/(p0+0.02)})

                for ln in noise_levels:
                    for mr in missing_levels:
                        for ir in imbalance_levels:
                            pn = metric_value(df, dataset, model, seed, ln, 0, "natural", metric)
                            pm = metric_value(df, dataset, model, seed, 0, mr, "natural", metric)
                            pi = metric_value(df, dataset, model, seed, 0, 0, ir, metric)
                            pall = metric_value(df, dataset, model, seed, ln, mr, ir, metric)
                            if not any(pd.isna(v) for v in [pn, pm, pi, pall]):
                                dn, dm, di, dall = p0-pn, p0-pm, p0-pi, p0-pall
                                exp = dn + dm + di
                                out.append({"combination": "all_three_excess_over_singles", "dataset_name": dataset, "model_name": model, "seed": seed, "metric": metric,
                                            "actual_drop": dall, "expected_drop": exp, "interaction": dall-exp,
                                            "bounded_interaction": dall-min(exp,p0), "relative_clean_interaction": (dall-exp)/(p0+0.02)})
    return pd.DataFrame(out)

def summarize_interactions(inter):
    rows = []
    for (metric, combo), g in inter.groupby(["metric", "combination"]):
        ds = g.groupby("dataset_name")["interaction"].mean()
        rows.append({
            "metric": metric, "combination": combo,
            "n_rows": int(len(g)), "n_datasets": int(g["dataset_name"].nunique()),
            "mean_interaction": float(g["interaction"].mean()),
            "dataset_cluster_mean": float(ds.mean()),
            "dataset_cluster_std": float(ds.std()),
        })
    return pd.DataFrame(rows)

def run_analyze():
    out_dir = ensure_dir(Path(OUT_ROOT) / "analysis")
    raw_files = sorted((Path(OUT_ROOT) / "external").glob("raw_results_external_part_*.csv"))

    if raw_files:
        raw = pd.concat([pd.read_csv(f) for f in raw_files], ignore_index=True).drop_duplicates("run_key")
        raw.to_csv(out_dir / "raw_external_all_unique.csv", index=False)
        print("External rows:", len(raw))

        all_inter = []
        for metric in ["macro_f1", "balanced_accuracy", "accuracy", "mcc", "minority_recall"]:
            inter = compute_interactions(raw, metric=metric)
            inter.to_csv(out_dir / f"external_interactions_{metric}.csv", index=False)
            all_inter.append(inter)

        all_inter = pd.concat(all_inter, ignore_index=True)
        all_inter.to_csv(out_dir / "external_interactions_all_metrics.csv", index=False)

        summary = summarize_interactions(all_inter)
        summary.to_csv(out_dir / "external_interaction_summary.csv", index=False)
        display(summary)
    else:
        print("No external raw files found.")

    zip_path = "/kaggle/working/q1_defense_openml_outputs.zip"
    if Path(zip_path).exists():
        Path(zip_path).unlink()
    shutil.make_archive("/kaggle/working/q1_defense_openml_outputs", "zip", OUT_ROOT)
    print("Final ZIP:", zip_path)


In [ ]:

# =========================
# RUN SELECTED STAGE
# =========================

manifest_path = Path(OUT_ROOT) / "manifest" / "openml_manifest_valid.csv"

if RUN_STAGE == "build_openml":
    build_openml_manifest()

elif RUN_STAGE == "external":
    assert manifest_path.exists(), f"Manifest not found: {manifest_path}. First run RUN_STAGE='build_openml'."
    run_external(manifest_path)

elif RUN_STAGE == "small":
    assert manifest_path.exists(), f"Manifest not found: {manifest_path}. First run RUN_STAGE='build_openml'."
    run_small(manifest_path)

elif RUN_STAGE == "analyze":
    run_analyze()

else:
    raise ValueError("Unknown RUN_STAGE.")

# Always create/update ZIP at end of each stage.
zip_path = "/kaggle/working/q1_defense_openml_outputs.zip"
if Path(zip_path).exists():
    Path(zip_path).unlink()
shutil.make_archive("/kaggle/working/q1_defense_openml_outputs", "zip", OUT_ROOT)
print("Saved ZIP:", zip_path)
